# Toxic or Just Tilted? — End-to-end demo

This notebook walks through the full pipeline: data loading, training Models A/B/C, evaluating on three test conditions, and validating the player reputation system. It runs end-to-end in roughly 3 minutes on a CPU.

**Prerequisites**
1. `pip install -r requirements.txt`
2. Datasets cloned into `data/CONDA` and `data/hate-speech-and-offensive-language` (see README for `git clone` commands)


## 1. Load and inspect the datasets

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so we can import src.*

from src.data_loader import make_splits
splits = make_splits()
for name, split in splits.items():
    print(f'{name:20s}  n={len(split.y):,}  toxic={int(split.y.sum()):,}  ({split.y.mean():.1%})')

## 2. Train Models A and B (the standard-approach baselines)

Both are TF-IDF + Logistic Regression. Model A is trained on Davidson (general internet toxicity); Model B is trained on CONDA (in-domain Dota 2 chat).

In [ ]:
from src.train import train_all
models = train_all()
print('Trained:', list(models.keys()))

## 3. The cross-domain transfer failure

Each model performs well in-domain but degrades sharply when applied to the other domain. This is the headline finding for §4.1 of the report.

In [ ]:
from src.evaluate import build_eval_matrix
matrix = build_eval_matrix()
matrix.round(3)

## 4. Inspect what each model has learned

Logistic regression makes its learned features directly inspectable. The same word can have opposite roles in the two models.

In [ ]:
import numpy as np
from src.train import load_pipeline

for name, slug in [('Model A (Davidson)', 'model_a_davidson'), ('Model B (CONDA)', 'model_b_conda')]:
    pipe = load_pipeline(slug)
    vec = pipe.named_steps['tfidf']
    clf = pipe.named_steps['clf']
    feats = np.array(vec.get_feature_names_out())
    coefs = clf.coef_[0]
    top = np.argsort(coefs)[-10:][::-1]
    print(f'\n{name} — top 10 toxic-leaning features:')
    for i in top:
        print(f'  {feats[i]:25s}  {coefs[i]:+.2f}')

## 5. The adversarial obfuscation failure

Both models catch only ~17% of obfuscated slurs. Our regex-based normalizer pre-processes inputs to undo common obfuscation tricks.

In [ ]:
from src.adversarial import normalize, contains_slur, build_adversarial_probes

examples = [
    'r e t a r d',          # spacing
    'r3t4rd',               # leet
    'kysssssss',            # repetition
    'f.a.g.g.o.t',          # symbols
    'good game',            # benign control
    'h3ll0 t34m',           # benign leet
]
for e in examples:
    print(f'  {e!r:30s}  ->  {normalize(e)!r:25s}  slur_detected={contains_slur(e)}')

probes = build_adversarial_probes()
print(f'\nBuilt {len(probes)} adversarial probes ({sum(p.expected_toxic for p in probes)} toxic, {sum(1 for p in probes if not p.expected_toxic)} controls)')

## 6. Train Model C (multi-class + context + integrated normalization)

In [ ]:
from src.model_c import train_model_c, evaluate_model_c
pipe_c, train_df, test_df = train_model_c()
evaluate_model_c(pipe_c, test_df)

## 7. Full head-to-head: Models A, B, C

Model C wins big on adversarial robustness and in-domain F1; loses on identity-statement FPR. The honest tradeoff is the centerpiece of §4.4 and §5 of the report.

In [ ]:
from src.benchmark_all import main as benchmark_all
benchmark_all()

## 8. Player reputation: temporal validation

For each player with ≥10 messages, compute a reputation score from the first 70% of their messages (using Model C's per-message severities) and check whether it predicts the toxicity rate of their final 30%.

In [ ]:
from src.reputation import main as reputation_main
reputation_main()

## 9. Generate all figures

Saves PNGs to `figures/`.

In [ ]:
from src.visualize import main as viz_v1
from src.visualize_v2 import make_all as viz_v2
viz_v1()
viz_v2()

---

Full writeup in [`../report/REPORT.md`](../report/REPORT.md).
